# First-Order Logic Inference and Resolution

## Aim
To represent knowledge using First-Order Logic (FOL), answer queries through backward chaining, examine the validity of arguments, and prove conclusions using resolution with unification.

## Learning objectives
1. Represent facts and rules using predicates, variables, and constants.
2. Apply backward chaining to Horn-clause rules.
3. Distinguish a valid conclusion from an invalid one using a countermodel.
4. Represent existential statements with an explicit witness.
5. Convert implication rules into clause form for resolution.
6. Implement first-order unification, standardizing variables apart, and resolution.

## Scope and assumptions
This notebook is an educational, restricted FOL demonstrator. The backward-chaining engine handles positive Horn clauses. The resolution engine accepts clauses already expressed in conjunctive normal form (CNF); it does **not** automatically parse arbitrary English or convert every possible FOL formula to CNF. The implementation supports predicate literals with constants and variables, but not nested function terms or equality reasoning.

## 1. First-Order Logic: basic representation

A predicate describes a property or relationship. A constant names an individual, and a variable ranges over individuals.

Examples:

- `Human(Socrates)` — Socrates is human.
- `∀x (Human(x) → Mortal(x))` — every human is mortal.
- `Student(Rahul) ∧ Studies(Rahul) → Pass(Rahul)`.

For resolution, implications are rewritten as disjunctions of literals:

`P → Q` becomes `¬P ∨ Q`.

A proof by resolution adds the negation of the query to the knowledge base. Deriving the empty clause (contradiction) establishes that the query follows from the clauses. First-order resolution uses unification to match complementary literals and variables must be standardized apart before clauses are combined. citeturn876371search13turn876371search1

## 2. Represent facts and Horn-clause rules

In [1]:
# Facts are ground predicates: (predicate_name, argument)
facts = {
    ("Human", "Socrates"),
    ("Student", "Rahul"),
    ("Studies", "Rahul")
}

# Each rule has a list of premises and one conclusion.
# Variables are written with a leading '?'.
rules = [
    {
        "premises": [("Human", "?x")],
        "conclusion": ("Mortal", "?x")
    },
    {
        "premises": [("Mortal", "?x")],
        "conclusion": ("Dies", "?x")
    },
    {
        "premises": [("Student", "?x"), ("Studies", "?x")],
        "conclusion": ("Pass", "?x")
    }
]

print("FACTS")
for fact in sorted(facts):
    print(fact)

print("\nRULES")
for rule in rules:
    print(rule["premises"], "->", rule["conclusion"])

FACTS
('Human', 'Socrates')
('Student', 'Rahul')
('Studies', 'Rahul')

RULES
[('Human', '?x')] -> ('Mortal', '?x')
[('Mortal', '?x')] -> ('Dies', '?x')
[('Student', '?x'), ('Studies', '?x')] -> ('Pass', '?x')


## 3. Backward chaining with unification
The query is matched against rule conclusions. The resulting variable substitutions are then used to prove every premise.

In [2]:
def is_variable(term):
    return isinstance(term, str) and term.startswith("?")

def apply_substitution(atom, substitution):
    resolved = []
    for term in atom:
        while is_variable(term) and term in substitution:
            term = substitution[term]
        resolved.append(term)
    return tuple(resolved)

def unify_atoms(pattern, ground_atom, substitution=None):
    if substitution is None:
        substitution = {}

    pattern = apply_substitution(pattern, substitution)

    if len(pattern) != len(ground_atom) or pattern[0] != ground_atom[0]:
        return None

    result = dict(substitution)

    for left, right in zip(pattern[1:], ground_atom[1:]):
        left = apply_substitution((left,), result)[0]
        right = apply_substitution((right,), result)[0]

        if is_variable(left):
            result[left] = right
        elif is_variable(right):
            result[right] = left
        elif left != right:
            return None

    return result

def backward_chain(query, active_goals=None, depth=0, max_depth=30):
    if active_goals is None:
        active_goals = set()

    query = tuple(query)

    if query in facts:
        return True

    if depth >= max_depth or query in active_goals:
        return False

    next_active = active_goals | {query}

    for rule in rules:
        substitution = unify_atoms(rule["conclusion"], query)

        if substitution is None:
            continue

        grounded_premises = [
            apply_substitution(premise, substitution)
            for premise in rule["premises"]
        ]

        if all(
            backward_chain(premise, next_active, depth + 1, max_depth)
            for premise in grounded_premises
        ):
            return True

    return False

queries = [
    ("Mortal", "Socrates"),
    ("Dies", "Socrates"),
    ("Pass", "Rahul")
]

for query in queries:
    print(f"{query}: {backward_chain(query)}")

('Mortal', 'Socrates'): True
('Dies', 'Socrates'): True
('Pass', 'Rahul'): True


## 4. Example: all humans are mortal
FOL: `∀x (Human(x) → Mortal(x))` and `Human(Socrates)`. Query: `Mortal(Socrates)`.

In [3]:
print("Premise 1: For every x, Human(x) implies Mortal(x).")
print("Premise 2: Human(Socrates).")
print("Query: Mortal(Socrates)?")

result = backward_chain(("Mortal", "Socrates"))
print("Backward-chaining result:", "PROVED" if result else "NOT PROVED")

Premise 1: For every x, Human(x) implies Mortal(x).
Premise 2: Human(Socrates).
Query: Mortal(Socrates)?
Backward-chaining result: PROVED


## 5. Validity check: dogs and pets

Given:
- All dogs are animals: `∀x (Dog(x) → Animal(x))`
- Some animals are pets: `∃x (Animal(x) ∧ Pet(x))`

Proposed conclusion: `∃x (Dog(x) ∧ Pet(x))`

The conclusion does **not** follow. The animal that is a pet need not be a dog. The following finite interpretation is a countermodel: Fido is a dog and an animal; Fluffy is an animal and a pet, but is not a dog.

In [4]:
# A countermodel: all listed sets describe the same small domain.
domain = {"Fido", "Fluffy"}
dogs = {"Fido"}
animals = {"Fido", "Fluffy"}
pets = {"Fluffy"}

premise_1 = dogs.issubset(animals)
premise_2 = len(animals & pets) > 0
conclusion = len(dogs & pets) > 0

print("All dogs are animals:", premise_1)
print("Some animals are pets:", premise_2)
print("Some dogs are pets:", conclusion)

if premise_1 and premise_2 and not conclusion:
    print("\nThe argument is INVALID: the premises are true while the conclusion is false.")
else:
    print("\nThis interpretation does not serve as a countermodel.")

All dogs are animals: True
Some animals are pets: True
Some dogs are pets: False

The argument is INVALID: the premises are true while the conclusion is false.


## 6. Existential statement: all humans love someone

Statement: `∀x (Human(x) → ∃y Loves(x, y))`

Fact: `Human(John)`

The existential quantifier means that at least one person is loved by John. To represent this in a small executable knowledge base, introduce a named witness constant `SomeoneLovedByJohn`. This witness is a modelling device for the example; it is not a claim that the person's real-world identity is known.

In [5]:
# Facts and a witness for the existential conclusion
human_facts = {"John", "Socrates"}
loves_facts = {
    ("Loves", "John", "SomeoneLovedByJohn")
}

def loves_someone(person):
    return any(
        predicate == "Loves" and subject == person
        for predicate, subject, obj in loves_facts
    )

print("Human(John):", "John" in human_facts)
print("Query: Does John love someone?", loves_someone("John"))

if loves_someone("John"):
    witness = next(obj for predicate, subject, obj in loves_facts
                   if predicate == "Loves" and subject == "John")
    print("Witness used:", witness)

Human(John): True
Query: Does John love someone? True
Witness used: SomeoneLovedByJohn


## 7. Resolution: clause representation

For the proof:

- `∀x (Human(x) → Mortal(x))` becomes `¬Human(x) ∨ Mortal(x)`.
- `∀x (Mortal(x) → Dies(x))` becomes `¬Mortal(x) ∨ Dies(x)`.
- Fact: `Human(Socrates)`.
- Negated query: `¬Dies(Socrates)`.

Resolution attempts to derive the empty clause `□`. A literal is represented as `(is_negated, predicate, arguments)`. Variables begin with `?`.

In [6]:
# Literal format: (is_negated, predicate_name, tuple_of_arguments)
def L(negated, predicate, *args):
    return (negated, predicate, tuple(args))

resolution_clauses = [
    [L(False, "Human", "Socrates")],
    [L(True, "Human", "?x"), L(False, "Mortal", "?x")],
    [L(True, "Mortal", "?y"), L(False, "Dies", "?y")],
    [L(True, "Dies", "Socrates")]  # negation of the query
]

def format_literal(literal):
    negated, predicate, args = literal
    text = f"{predicate}({', '.join(args)})"
    return ("¬" if negated else "") + text

def format_clause(clause):
    if not clause:
        return "□"
    return " ∨ ".join(format_literal(lit) for lit in clause)

for index, clause in enumerate(resolution_clauses, start=1):
    print(f"C{index}: {format_clause(clause)}")

C1: Human(Socrates)
C2: ¬Human(?x) ∨ Mortal(?x)
C3: ¬Mortal(?y) ∨ Dies(?y)
C4: ¬Dies(Socrates)


## 8. Implement first-order unification
The unifier finds a substitution that makes two predicate argument lists identical. This implementation includes variable dereferencing and an occurs check for the supported constant/variable term representation.

In [7]:
def dereference(term, substitution):
    while is_variable(term) and term in substitution:
        term = substitution[term]
    return term

def occurs(variable, term, substitution):
    return dereference(term, substitution) == variable

def unify_terms(left, right, substitution):
    left = dereference(left, substitution)
    right = dereference(right, substitution)

    if left == right:
        return substitution

    if is_variable(left):
        if occurs(left, right, substitution):
            return None
        substitution[left] = right
        return substitution

    if is_variable(right):
        if occurs(right, left, substitution):
            return None
        substitution[right] = left
        return substitution

    return None

def unify_literals(lit1, lit2, substitution=None):
    if substitution is None:
        substitution = {}

    neg1, pred1, args1 = lit1
    neg2, pred2, args2 = lit2

    # Only complementary literals of the same predicate/arity can resolve.
    if neg1 == neg2 or pred1 != pred2 or len(args1) != len(args2):
        return None

    result = dict(substitution)

    for term1, term2 in zip(args1, args2):
        result = unify_terms(term1, term2, result)
        if result is None:
            return None

    return result

# Example: Human(?x) unifies with Human(Socrates).
example_substitution = unify_literals(
    L(False, "Human", "?x"),
    L(True, "Human", "Socrates")
)
print("Unifier:", example_substitution)

Unifier: {'?x': 'Socrates'}


## 9. Standardize variables apart and apply substitutions

In [8]:
def standardize_apart(clause, suffix):
    rename = {}

    def rename_term(term):
        if is_variable(term):
            if term not in rename:
                rename[term] = f"?v{suffix}_{term[1:]}"
            return rename[term]
        return term

    return [
        (negated, predicate, tuple(rename_term(t) for t in args))
        for negated, predicate, args in clause
    ]

def apply_literal_substitution(literal, substitution):
    negated, predicate, args = literal
    new_args = tuple(dereference(term, substitution) for term in args)
    return (negated, predicate, new_args)

def is_tautology(clause):
    literal_set = set(clause)
    for negated, predicate, args in literal_set:
        if (not negated, predicate, args) in literal_set:
            return True
    return False

## 10. First-order resolution engine
The engine repeatedly resolves pairs of clauses, adds new resolvents, and stops if it derives the empty clause. A maximum-clause limit protects the demonstration from unbounded search on examples that do not yield a quick proof.

In [9]:
def resolve_pair(clause1, clause2, suffix):
    c1 = standardize_apart(clause1, f"{suffix}a")
    c2 = standardize_apart(clause2, f"{suffix}b")
    resolvents = []

    for i, lit1 in enumerate(c1):
        for j, lit2 in enumerate(c2):
            substitution = unify_literals(lit1, lit2)

            if substitution is None:
                continue

            remaining = c1[:i] + c1[i+1:] + c2[:j] + c2[j+1:]
            resolvent = [
                apply_literal_substitution(lit, substitution)
                for lit in remaining
            ]

            # Remove duplicate literals while preserving order.
            deduplicated = []
            for lit in resolvent:
                if lit not in deduplicated:
                    deduplicated.append(lit)

            if not is_tautology(deduplicated):
                resolvents.append(deduplicated)

    return resolvents

def first_order_resolution(initial_clauses, max_clauses=500):
    clauses = [list(clause) for clause in initial_clauses]
    known = {frozenset(clause) for clause in clauses}
    trace = []
    pair_number = 0

    if frozenset() in known:
        return True, trace

    while len(clauses) < max_clauses:
        new_clauses = []

        for i in range(len(clauses)):
            for j in range(i + 1, len(clauses)):
                pair_number += 1
                resolvents = resolve_pair(
                    clauses[i], clauses[j], pair_number
                )

                for resolvent in resolvents:
                    key = frozenset(resolvent)

                    if key in known:
                        continue

                    known.add(key)
                    new_clauses.append(resolvent)
                    trace.append(
                        (clauses[i], clauses[j], resolvent)
                    )

                    if len(resolvent) == 0:
                        return True, trace

                    if len(clauses) + len(new_clauses) >= max_clauses:
                        return False, trace

        if not new_clauses:
            return False, trace

        clauses.extend(new_clauses)

    return False, trace

proved, proof_trace = first_order_resolution(resolution_clauses)

print("Resolution derivation:")
for step, (parent1, parent2, resolvent) in enumerate(proof_trace, start=1):
    print(f"Step {step}:")
    print("  Parent 1:", format_clause(parent1))
    print("  Parent 2:", format_clause(parent2))
    print("  Resolvent:", format_clause(resolvent))

print("\nQuery: Dies(Socrates)")
print("Result:", "PROVED (empty clause derived)" if proved else "NOT PROVED")

Resolution derivation:
Step 1:
  Parent 1: Human(Socrates)
  Parent 2: ¬Human(?x) ∨ Mortal(?x)
  Resolvent: Mortal(Socrates)
Step 2:
  Parent 1: ¬Human(?x) ∨ Mortal(?x)
  Parent 2: ¬Mortal(?y) ∨ Dies(?y)
  Resolvent: ¬Human(?v4b_y) ∨ Dies(?v4b_y)
Step 3:
  Parent 1: ¬Mortal(?y) ∨ Dies(?y)
  Parent 2: ¬Dies(Socrates)
  Resolvent: ¬Mortal(Socrates)
Step 4:
  Parent 1: Human(Socrates)
  Parent 2: ¬Human(?v4b_y) ∨ Dies(?v4b_y)
  Resolvent: Dies(Socrates)
Step 5:
  Parent 1: ¬Human(?x) ∨ Mortal(?x)
  Parent 2: ¬Mortal(?y) ∨ Dies(?y)
  Resolvent: ¬Human(?v13b_y) ∨ Dies(?v13b_y)
Step 6:
  Parent 1: ¬Human(?x) ∨ Mortal(?x)
  Parent 2: ¬Mortal(Socrates)
  Resolvent: ¬Human(Socrates)
Step 7:
  Parent 1: Mortal(Socrates)
  Parent 2: ¬Mortal(Socrates)
  Resolvent: □

Query: Dies(Socrates)
Result: PROVED (empty clause derived)


## 11. Additional unification test with multiple predicates

This test demonstrates that variables can be matched across predicates and that a mismatch in constants prevents unification.

In [10]:
test_1 = unify_literals(
    L(False, "Parent", "?x", "?y"),
    L(True, "Parent", "Alice", "Bob")
)

test_2 = unify_literals(
    L(False, "Parent", "Alice", "?y"),
    L(True, "Parent", "Charlie", "Bob")
)

print("Parent(?x, ?y) with Parent(Alice, Bob):", test_1)
print("Parent(Alice, ?y) with Parent(Charlie, Bob):", test_2)

Parent(?x, ?y) with Parent(Alice, Bob): {'?x': 'Alice', '?y': 'Bob'}
Parent(Alice, ?y) with Parent(Charlie, Bob): None


## 12. Observations

1. Backward chaining starts with a query and works backward through rules to known facts.
2. The Socrates example derives `Mortal(Socrates)` and `Dies(Socrates)`.
3. The dogs-and-pets conclusion is invalid because the animal that is a pet is not necessarily a dog.
4. An existential statement requires at least one witness; the notebook represents that witness explicitly.
5. First-order resolution uses unification to match complementary literals containing variables.
6. The resolution proof derives the empty clause after adding the negation of `Dies(Socrates)`.

## Conclusion

The notebook demonstrates First-Order Logic representation, backward chaining, validity checking, existential witness representation, unification, and first-order resolution. The examples show how a knowledge base can be used to derive conclusions and how a proposed conclusion can be tested against the premises.

## Limitations

- English-to-FOL translation is not automated; formulas are entered explicitly.
- The backward-chaining component handles positive Horn-clause rules.
- The resolution component accepts clauses already in CNF. It does not implement a complete automatic CNF converter, Skolemization pipeline, equality resolution, or nested function terms.
- First-order resolution is not guaranteed to terminate when a query is not entailed. The implementation therefore uses a clause limit for safety.

## References

- Stanford University, *Introduction to Logic: Resolution and Unification*: https://logic.stanford.edu/intrologic/extras/resolution.html
- Oregon State University, *First-Order Logic Inference*: https://web.engr.oregonstate.edu/~afern/GOFAI/notes/first-order-logic-inference.pdf